# 缺失值填充方法

本notebook专门介绍几种高级缺失值填充方法：

1. 条件平均值填充

2. 热卡填充（Hot deck imputation）

3. KNN填充

4. 回归填充（Regression）

5. 多重插补（Multiple Imputation）

## 1. 条件平均值填充

与其相似的另一种方法叫**条件平均值填充法**（Conditional Mean Completer）。在该方法中，用于求平均的值并不是从数据集的所有对象中取，而是从与该对象具有**相同决策属性值**的对象中取得。

In [1]:
import pandas as pd
import numpy as np

# 构造示例数据
df = pd.DataFrame({
    "用户ID": [1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008, 1009, 1010],
    "姓名": ["张三", "李四", "王五", "赵六", "孙七", "周八", "吴九", "郑十", "钱十一", "陈十二"],
    "性别": ["男", "女", "男", "女", "男", "女", "男", "女", "男", "女"],
    "年龄": [25, 30, np.nan, 45, 30, 28, np.nan, 32, 40, 35],
    "城市": ["beijing", "Shanghai", "guangzhou", "shenzhen", "Shanghai", "Beijing", "hangzhou", "Nanjing", "shenzhen", "Beijing"],
    "年收入": [80000, 120000, 75000, 95000, 120000, 110000, 98000, 105000, 88000, 92000],
    "上次消费金额": [200, 500, 300, np.nan, 500, 400, 250, 350, 280, 320]
})

print("原始数据：")
print(df)

# 按【性别】分组，填充【年龄】缺失值
df["年龄"] = df.groupby("性别")["年龄"].transform(
    lambda x: x.fillna(x.mean())
)

# 按【性别+城市】双条件分组，填充年龄缺失值
df["年龄"] = df.groupby(["性别", "城市"])["年龄"].transform(
    lambda x: x.fillna(x.mean())
)

# 按【城市】分组，填充【上次消费金额】缺失值
df["上次消费金额"] = df.groupby("城市")["上次消费金额"].transform(
    lambda x: x.fillna(x.mean())
)

print("\n条件平均值填充后的数据：")
print(df)

原始数据：
   用户ID   姓名 性别    年龄         城市     年收入  上次消费金额
0  1001   张三  男  25.0    beijing   80000   200.0
1  1002   李四  女  30.0   Shanghai  120000   500.0
2  1003   王五  男   NaN  guangzhou   75000   300.0
3  1004   赵六  女  45.0   shenzhen   95000     NaN
4  1005   孙七  男  30.0   Shanghai  120000   500.0
5  1006   周八  女  28.0    Beijing  110000   400.0
6  1007   吴九  男   NaN   hangzhou   98000   250.0
7  1008   郑十  女  32.0    Nanjing  105000   350.0
8  1009  钱十一  男  40.0   shenzhen   88000   280.0
9  1010  陈十二  女  35.0    Beijing   92000   320.0

条件平均值填充后的数据：
   用户ID   姓名 性别         年龄         城市     年收入  上次消费金额
0  1001   张三  男  25.000000    beijing   80000   200.0
1  1002   李四  女  30.000000   Shanghai  120000   500.0
2  1003   王五  男  31.666667  guangzhou   75000   300.0
3  1004   赵六  女  45.000000   shenzhen   95000   280.0
4  1005   孙七  男  30.000000   Shanghai  120000   500.0
5  1006   周八  女  28.000000    Beijing  110000   400.0
6  1007   吴九  男  31.666667   hangzhou   98000   250.0
7  1008  

## 2. 热卡填充（Hot deck imputation，或就近补齐）
-------------------------------------------------------------------

对于一个包含空值的对象，热卡填充法在完整数据中找到**一个**与它最相似的对象，然后用这个**相似对象的值来进行填充**。

不同的问题可能会选用不同的标准来对相似进行判定。该方法概念上很简单，且利用了数据间的关系来进行空值估计。

这个方法的缺点在于难以定义相似标准，**主观因素较多**。

In [2]:
import pandas as pd
import numpy as np
from sklearn.metrics import pairwise_distances
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# 构造示例数据
df = pd.DataFrame({
    "用户ID": [1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008, 1009, 1010],
    "姓名": ["张三", "李四", "王五", "赵六", "孙七", "周八", "吴九", "郑十", "钱十一", "陈十二"],
    "性别": ["男", "女", "男", "女", "男", "女", "男", "女", "男", "女"],
    "年龄": [25, 30, np.nan, 45, 30, 28, np.nan, 32, 40, 35],
    "城市": ["beijing", "Shanghai", "guangzhou", "shenzhen", "Shanghai", "Beijing", "hangzhou", "Nanjing", "shenzhen", "Beijing"],
    "年收入": [80000, 120000, 75000, 95000, 120000, 110000, 98000, 105000, 88000, 92000],
    "上次消费金额": [200, 500, 300, np.nan, 500, 400, 250, 350, 280, 320]
})

print("原始数据：")
print(df)


df['城市'] = df['城市'].str.lower()

# 3. 定义热卡填充核心函数
def hot_deck_impute(df, target_col, sim_features):
    """
    热卡填充函数
    :param df: 原始数据集
    :param target_col: 需要填充的目标列
    :param sim_features: 用于计算样本相似度的特征列表（无缺失）
    :return: 填充后的目标列
    """
    # 复制数据，避免修改原数据
    df_impute = df.copy()
    # 筛选：完整样本（目标列无缺失）+ 缺失样本（目标列有缺失）
    complete_samples = df_impute[df_impute[target_col].notna()]
    missing_samples = df_impute[df_impute[target_col].isna()]

    if len(missing_samples) == 0:
        return df_impute[target_col]

    # 对相似度特征做：分类特征独热编码 + 数值特征标准化
    # 提取相似度特征
    sim_data = df_impute[sim_features]
    # 分离分类/数值特征
    cat_cols = sim_data.select_dtypes(include=['object', 'string']).columns.tolist()
    num_cols = sim_data.select_dtypes(include=['int64', 'float64']).columns.tolist()

    # 独热编码分类特征
    encoder = OneHotEncoder(sparse_output=False, drop='first')
    cat_encoded = encoder.fit_transform(sim_data[cat_cols])
    cat_df = pd.DataFrame(cat_encoded, columns=encoder.get_feature_names_out(cat_cols), index=sim_data.index)

    # 标准化数值特征
    scaler = StandardScaler()
    num_scaled = scaler.fit_transform(sim_data[num_cols])
    num_df = pd.DataFrame(num_scaled, columns=num_cols, index=sim_data.index)

    # 合并最终相似度矩阵
    sim_matrix = pd.concat([cat_df, num_df], axis=1)

    # 遍历所有缺失样本，填充最相似完整样本的值
    for missing_idx in missing_samples.index:
        # 缺失样本的特征向量
        missing_vec = sim_matrix.loc[[missing_idx]]
        # 完整样本的特征向量
        complete_vec = sim_matrix.loc[complete_samples.index]
        # 计算欧氏距离（找最近邻）
        distances = pairwise_distances(missing_vec, complete_vec, metric='euclidean')[0]
        # 最相似的完整样本索引
        most_similar_idx = complete_samples.index[np.argmin(distances)]
        # 用最相似样本的值填充
        df_impute.loc[missing_idx, target_col] = df_impute.loc[most_similar_idx, target_col]

    return df_impute[target_col]

# 4. 配置相似度特征（无缺失的核心特征：性别、城市、年收入）
similarity_features = ['性别', '城市', '年收入']

# 5. 执行热卡填充
df['年龄'] = hot_deck_impute(df, '年龄', similarity_features)
df['上次消费金额'] = hot_deck_impute(df, '上次消费金额', similarity_features)

print("\n热卡填充后的数据：")
print(df)


原始数据：
   用户ID   姓名 性别    年龄         城市     年收入  上次消费金额
0  1001   张三  男  25.0    beijing   80000   200.0
1  1002   李四  女  30.0   Shanghai  120000   500.0
2  1003   王五  男   NaN  guangzhou   75000   300.0
3  1004   赵六  女  45.0   shenzhen   95000     NaN
4  1005   孙七  男  30.0   Shanghai  120000   500.0
5  1006   周八  女  28.0    Beijing  110000   400.0
6  1007   吴九  男   NaN   hangzhou   98000   250.0
7  1008   郑十  女  32.0    Nanjing  105000   350.0
8  1009  钱十一  男  40.0   shenzhen   88000   280.0
9  1010  陈十二  女  35.0    Beijing   92000   320.0

热卡填充后的数据：
   用户ID   姓名 性别    年龄         城市     年收入  上次消费金额
0  1001   张三  男  25.0    beijing   80000   200.0
1  1002   李四  女  30.0   shanghai  120000   500.0
2  1003   王五  男  25.0  guangzhou   75000   300.0
3  1004   赵六  女  45.0   shenzhen   95000   320.0
4  1005   孙七  男  30.0   shanghai  120000   500.0
5  1006   周八  女  28.0    beijing  110000   400.0
6  1007   吴九  男  35.0   hangzhou   98000   250.0
7  1008   郑十  女  32.0    nanjing  105000   350.0
8  

#### 常见错误

1. 分类特征编码错误（最严重）
代码用 LabelEncoder 编码城市、性别（无序分类特征）
LabelEncoder 会把城市转为 0,1,2,3...，让欧氏距离误以为：
北京(0) < 上海(1) < 广州(2)，给无序特征强加了大小顺序
后果：相似度计算完全失真，找的 “最相似样本” 根本不相似
✅ 标准代码：用 OneHotEncoder 编码无序分类特征，无顺序偏差，计算科学。
2. 未做数值特征标准化
数据中：
年收入：80000~120000（极大数值）
性别 / 城市编码：0/1（极小数值）
欧氏距离会完全被年收入主导，性别、城市对相似度的影响几乎为 0。
✅ 标准代码：对数值特征标准化，消除量纲影响，所有特征公平参与相似度计算。
3. 包含无效干扰特征
代码自动用所有列计算距离，包含：用户ID、姓名（唯一标识，无业务意义）比如用户 ID 1001 和 1002 会因为 ID 数字接近被判定为相似，完全错误。
✅ 标准代码：手动筛选核心相似度特征（性别、城市、年收入），剔除干扰项。
4. 数据未预处理（城市大小写不统一）
原始数据中：beijing / Beijing、Shanghai / shanghai代码会把它们当成两个不同城市，分类错误。
✅ 标准代码：统一城市小写，保证分类准确性。
5. 填充连锁误差
先填充年龄 → 用填充后的年龄再去计算消费金额的相似度填充的估算值会二次干扰结果，误差叠加。
✅ 标准代码：始终用原始完整特征计算相似度，无误差叠加。


## 3. KNN填充
--------------------------------------------------------------------

先根据欧式距离或相关分析来确定距离具有缺失数据样本最近的K个样本，将这K个值加权平均来估计该样本的缺失数据。

根据数据类型的不同，距离度量也不尽相同：
1、连续数据：最常用的距离度量有欧氏距离，曼哈顿距离以及余弦距离。
2、分类数据：汉明（Hamming）距离在这种情况比较常用。对于所有分类属性的取值，如果两个数据点的值不同，则距离加一。汉明距离实际上与属性间不同取值的数量一致。

KNN算法的一个明显缺点是，在分析大型数据集时会变得非常耗时，因为它会在整个数据集中搜索相似数据点。此外，在高维数据集中，最近与最远邻居之间的差别非常小，因此KNN的准确性会降低。

In [3]:
import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer
from sklearn.preprocessing import OneHotEncoder
import warnings
warnings.filterwarnings('ignore')

# 1. 原始数据
df = pd.DataFrame({
    "用户ID": [1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008, 1009, 1010],
    "姓名": ["张三", "李四", "王五", "赵六", "孙七", "周八", "吴九", "郑十", "钱十一", "陈十二"],
    "性别": ["男", "女", "男", "女", "男", "女", "男", "女", "男", "女"],
    "年龄": [25, 30, np.nan, 45, 30, 28, np.nan, 32, 40, 35],
    "城市": ["beijing", "Shanghai", "guangzhou", "shenzhen", "Shanghai", "Beijing", "hangzhou", "Nanjing", "shenzhen", "Beijing"],
    "年收入": [80000, 120000, 75000, 95000, 120000, 110000, 98000, 105000, 88000, 92000],
    "上次消费金额": [200, 500, 300, np.nan, 500, 400, 250, 350, 280, 320]
})
print("原始数据：")
print(df)
# ===================== 核心修正：预处理 + KNN填充 + 原始格式还原 =====================
# 2. 统一城市大小写（关键：避免Beijing/beijing识别为不同值）
df['城市'] = df['城市'].str.lower()
df_original = df.copy()  # 备份原始结构

# 3. 分离特征：分类特征（性别、城市）+ 数值特征（年龄、年收入、上次消费金额）
cat_cols = ['性别', '城市']       # 分类列
num_cols = ['年龄', '年收入', '上次消费金额']  # 数值列（含缺失值）

# 4. 独热编码分类特征（KNN专用，无顺序偏差）
encoder = OneHotEncoder(sparse_output=False, drop='first')
cat_encoded = encoder.fit_transform(df[cat_cols])
cat_df = pd.DataFrame(cat_encoded, columns=encoder.get_feature_names_out(cat_cols))

# 5. 合并编码后的分类特征 + 数值特征（用于KNN计算）
knn_data = pd.concat([cat_df, df[num_cols]], axis=1)

# 6. KNN填充缺失值（n_neighbors=3，小样本更合适）
imputer = KNNImputer(n_neighbors=3)
knn_filled = imputer.fit_transform(knn_data)
filled_df = pd.DataFrame(knn_filled, columns=knn_data.columns)

# 7. 逆变换 → 把编码还原为原始的【性别、城市】文字
cat_encoded_original = filled_df[encoder.get_feature_names_out(cat_cols)]
cat_original = encoder.inverse_transform(cat_encoded_original)  # 还原分类列
cat_original_df = pd.DataFrame(cat_original, columns=cat_cols)

# 8. 合并所有列，还原成【原始格式的DataFrame】
df_result = df_original.copy()
df_result[cat_cols] = cat_original_df  # 还原性别、城市
df_result[num_cols] = filled_df[num_cols]  # 填充后的数值列

# ===================== 输出最终结果 =====================
print("\nKNN填充后的数据：")
print(df_result.round(0).astype(int, errors='ignore'))  # 年龄/消费金额取整，更美观


原始数据：
   用户ID   姓名 性别    年龄         城市     年收入  上次消费金额
0  1001   张三  男  25.0    beijing   80000   200.0
1  1002   李四  女  30.0   Shanghai  120000   500.0
2  1003   王五  男   NaN  guangzhou   75000   300.0
3  1004   赵六  女  45.0   shenzhen   95000     NaN
4  1005   孙七  男  30.0   Shanghai  120000   500.0
5  1006   周八  女  28.0    Beijing  110000   400.0
6  1007   吴九  男   NaN   hangzhou   98000   250.0
7  1008   郑十  女  32.0    Nanjing  105000   350.0
8  1009  钱十一  男  40.0   shenzhen   88000   280.0
9  1010  陈十二  女  35.0    Beijing   92000   320.0

KNN填充后的数据：
   用户ID   姓名 性别  年龄         城市     年收入  上次消费金额
0  1001   张三  男  25    beijing   80000     200
1  1002   李四  女  30   shanghai  120000     500
2  1003   王五  男  33  guangzhou   75000     300
3  1004   赵六  女  45   shenzhen   95000     283
4  1005   孙七  男  30   shanghai  120000     500
5  1006   周八  女  28    beijing  110000     400
6  1007   吴九  男  37   hangzhou   98000     250
7  1008   郑十  女  32    nanjing  105000     350
8  1009  钱十一  男  40 

## 4. 回归填充（Regression）
---------------------------------------------------------------------

回归填充法是用已有数据建立回归模型，将缺失值作为因变量进行预测估计。

适用于缺失值与其他变量存在较强线性关系的情况。

In [4]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor  # 回归模型（核心）
from sklearn.preprocessing import OneHotEncoder     # 分类特征编码
import warnings
warnings.filterwarnings('ignore')

# 构造示例数据
df = pd.DataFrame({
    "用户ID": [1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008, 1009, 1010],
    "姓名": ["张三", "李四", "王五", "赵六", "孙七", "周八", "吴九", "郑十", "钱十一", "陈十二"],
    "性别": ["男", "女", "男", "女", "男", "女", "男", "女", "男", "女"],
    "年龄": [25, 30, np.nan, 45, 30, 28, np.nan, 32, 40, 35],
    "城市": ["beijing", "Shanghai", "guangzhou", "shenzhen", "Shanghai", "Beijing", "hangzhou", "Nanjing", "shenzhen", "Beijing"],
    "年收入": [80000, 120000, 75000, 95000, 120000, 110000, 98000, 105000, 88000, 92000],
    "上次消费金额": [200, 500, 300, np.nan, 500, 400, 250, 350, 280, 320]
})

print("原始数据：")
print(df)

# ===================== 3. 数据预处理（关键：统一格式+编码） =====================
df_process = df.copy()
# 统一城市大小写（消除beijing/Beijing差异）
df_process['城市'] = df_process['城市'].str.lower()

# 分类特征：性别、城市 → 独热编码（无序特征，无顺序偏差）
cat_features = ['性别', '城市']
encoder = OneHotEncoder(sparse_output=False, drop='first')
cat_encoded = encoder.fit_transform(df_process[cat_features])
cat_df = pd.DataFrame(cat_encoded, columns=encoder.get_feature_names_out(cat_features))

# 构建建模用数据集：编码特征 + 数值特征（剔除用户ID、姓名，无业务意义）
num_features = ['年龄', '年收入', '上次消费金额']
model_data = pd.concat([cat_df, df_process[num_features]], axis=1)

# ===================== 4. 定义回归填充通用函数 =====================
def regression_impute(model_data, target_col):
    """
    回归填充函数
    :param model_data: 编码后的建模数据
    :param target_col: 需要填充的目标列
    :return: 填充后的目标列数据
    """
    # 拆分：完整数据（训练集）、缺失数据（预测集）
    train = model_data[model_data[target_col].notna()]
    test = model_data[model_data[target_col].isna()]

    if len(test) == 0:
        return model_data[target_col]

    # 特征X（所有列排除目标列），目标y
    X_train = train.drop(target_col, axis=1)
    y_train = train[target_col]
    X_test = test.drop(target_col, axis=1)

    # 训练随机森林回归模型
    model = RandomForestRegressor(random_state=42, n_estimators=100)
    model.fit(X_train, y_train)

    # 预测缺失值
    model_data.loc[model_data[target_col].isna(), target_col] = model.predict(X_test)
    return model_data[target_col]

# ===================== 5. 执行回归填充 =====================
# 填充 年龄 缺失值
model_data['年龄'] = regression_impute(model_data, '年龄')
# 填充 上次消费金额 缺失值
model_data['上次消费金额'] = regression_impute(model_data, '上次消费金额')

# ===================== 6. 还原原始数据格式（保留性别、城市原始文字） =====================
df_final = df.copy()
df_final['年龄'] = model_data['年龄'].round(0).astype(int)  # 年龄取整
df_final['上次消费金额'] = model_data['上次消费金额'].round(0).astype(int)  # 消费金额取整
df_final['城市'] = df_process['城市']  # 统一大小写后的城市

print("\n回归填充后的数据：")
print(df_final)

原始数据：
   用户ID   姓名 性别    年龄         城市     年收入  上次消费金额
0  1001   张三  男  25.0    beijing   80000   200.0
1  1002   李四  女  30.0   Shanghai  120000   500.0
2  1003   王五  男   NaN  guangzhou   75000   300.0
3  1004   赵六  女  45.0   shenzhen   95000     NaN
4  1005   孙七  男  30.0   Shanghai  120000   500.0
5  1006   周八  女  28.0    Beijing  110000   400.0
6  1007   吴九  男   NaN   hangzhou   98000   250.0
7  1008   郑十  女  32.0    Nanjing  105000   350.0
8  1009  钱十一  男  40.0   shenzhen   88000   280.0
9  1010  陈十二  女  35.0    Beijing   92000   320.0

回归填充后的数据：
   用户ID   姓名 性别  年龄         城市     年收入  上次消费金额
0  1001   张三  男  25    beijing   80000     200
1  1002   李四  女  30   shanghai  120000     500
2  1003   王五  男  31  guangzhou   75000     300
3  1004   赵六  女  45   shenzhen   95000     314
4  1005   孙七  男  30   shanghai  120000     500
5  1006   周八  女  28    beijing  110000     400
6  1007   吴九  男  31   hangzhou   98000     250
7  1008   郑十  女  32    nanjing  105000     350
8  1009  钱十一  男  40  

## 方法对比

| 方法 | 优点 | 缺点 | 适用场景 |
|------|------|------|----------|
| 条件平均值填充 | 考虑了分类变量影响 | 需要有足够的分组样本 | 分组明显的缺失值 |
| 热卡填充 | 保持数据分布 | 相似度定义主观 | 小规模、结构化数据 |
| KNN填充 | 多维考虑，抗噪声 | 计算开销大 | 中等规模数据 |
| 回归填充 | 利用变量关系 | 假设线性关系 | 变量间有较强线性关系 |

## 5. 多重插补（Multiple Imputation）
-------------------------------------------------------------------

多重插补（Multiple Imputation, MI）是一种基于贝叶斯思想的缺失值处理方法。

**核心思想**：对每个缺失值进行**多次（通常3-5次）**插补，产生**多个完整数据集**，然后对每个数据集分别进行分析，最后将结果**汇总**得到最终估计。

**优势**：
- 考虑了缺失值的不确定性
- 能够反映模型参数的变异程度
- 比单次插补（single imputation）更准确、更可靠

**基本步骤**：
1. **插补（Imputation）**：对每个缺失值进行m次插补，生成m个完整数据集
2. **分析（Analysis）**：对每个完整数据集分别进行统计分析（如回归分析）
3. **汇总（Pooling）**：使用Rubin's rules将m个分析结果汇总为最终估计

**常用方法**：
- **MICE（Multivariate Imputation by Chained Equations）**：基于链式方程的多变量插补
- **PMM（预测均值匹配，Predictive Mean Matching）**
- **Bayesian MI（贝叶斯多重插补）**

**注意事项**：
 - 忽略缺失机制：MI 仅适用于 MAR/MCAR，若为MNAR（非随机缺失），需先通过敏感性分析评估偏差；
 - 过度插补：m 过大（>20）会增加计算成本，边际收益递减；
 - 结果合并错误：必须使用Rubin 规则，不可简单平均参数估计值。

In [5]:
import pandas as pd
import numpy as np
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.linear_model import LinearRegression, Ridge, BayesianRidge
from sklearn.ensemble import RandomForestRegressor
import warnings
warnings.filterwarnings('ignore')

# 构造示例数据
df = pd.DataFrame({
    "用户ID": [1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008, 1009, 1010],
    "姓名": ["张三", "李四", "王五", "赵六", "孙七", "周八", "吴九", "郑十", "钱十一", "陈十二"],
    "性别": ["男", "女", "男", "女", "男", "女", "男", "女", "男", "女"],
    "年龄": [25, 30, np.nan, 45, 30, 28, np.nan, 32, 40, 35],
    "城市": ["beijing", "Shanghai", "guangzhou", "shenzhen", "Shanghai", "Beijing", "hangzhou", "Nanjing", "shenzhen", "Beijing"],
    "年收入": [80000, 120000, 75000, 95000, 120000, 110000, 98000, 105000, 88000, 92000],
    "上次消费金额": [200, 500, 300, np.nan, 500, 400, 250, 350, 280, 320]
})

print("原始数据：")
print(df)
print(f"\n缺失值统计：\n{df.isnull().sum()}")

原始数据：
   用户ID   姓名 性别    年龄         城市     年收入  上次消费金额
0  1001   张三  男  25.0    beijing   80000   200.0
1  1002   李四  女  30.0   Shanghai  120000   500.0
2  1003   王五  男   NaN  guangzhou   75000   300.0
3  1004   赵六  女  45.0   shenzhen   95000     NaN
4  1005   孙七  男  30.0   Shanghai  120000   500.0
5  1006   周八  女  28.0    Beijing  110000   400.0
6  1007   吴九  男   NaN   hangzhou   98000   250.0
7  1008   郑十  女  32.0    Nanjing  105000   350.0
8  1009  钱十一  男  40.0   shenzhen   88000   280.0
9  1010  陈十二  女  35.0    Beijing   92000   320.0

缺失值统计：
用户ID      0
姓名        0
性别        0
年龄        2
城市        0
年收入       0
上次消费金额    1
dtype: int64


### 5.1 MICE（链式方程多变量插补）

MICE通过建立**条件模型**来迭代地填充缺失值。

**算法流程**：
1. 用均值初始化所有缺失值
2. 选择一个有缺失值的变量作为目标
3. 用其他变量预测目标变量的缺失值（建立回归模型）
4. 用预测值填补目标变量的缺失值
5. 重复步骤2-4，遍历所有有缺失值的变量
6. 重复整个过程多次（通常10-20次迭代）

sklearn的IterativeImputer实现了类似MICE的算法，默认使用BayesianRidge回归。

In [6]:
# ===================== 3. 数据预处理（关键步骤） =====================
df_process = df.copy()
# 统一城市大小写（消除 beijing/Beijing 差异，避免分类错误）
df_process['城市'] = df_process['城市'].str.lower()

# 划分特征类型
id_cols = ["用户ID", "姓名"]               # 唯一标识（不参与建模）
cat_cols = ["性别", "城市"]                # 分类特征（需要编码）
num_cols = ["年龄", "年收入", "上次消费金额"] # 数值特征（含缺失值）

# ===================== 4. 分类特征独热编码（无序特征专用） =====================
encoder = OneHotEncoder(sparse_output=False, drop='first')
cat_encoded = encoder.fit_transform(df_process[cat_cols])
cat_df = pd.DataFrame(cat_encoded, columns=encoder.get_feature_names_out(cat_cols))

# 构建MICE建模数据集（编码特征 + 数值特征）
mice_data = pd.concat([cat_df, df_process[num_cols]], axis=1)

# ===================== 5. MICE多重插补核心代码 =====================
# 初始化MICE插补器：使用随机森林回归作为预测器，稳定性更强
mice_imputer = IterativeImputer(
    estimator=RandomForestRegressor(random_state=42),
    random_state=42,
    max_iter=10  # 迭代10轮，保证填充收敛
)

# 执行多重插补
mice_filled = mice_imputer.fit_transform(mice_data)
filled_df = pd.DataFrame(mice_filled, columns=mice_data.columns)

# ===================== 6. 还原原始数据格式（无编码、纯业务字段） =====================
# 逆变换：将编码还原为 性别、城市 原始文字
cat_original = encoder.inverse_transform(filled_df[cat_df.columns])
cat_original_df = pd.DataFrame(cat_original, columns=cat_cols)

# 合并所有列，生成最终结果
df_final = df.copy()
df_final[cat_cols] = cat_original_df  # 还原分类特征
df_final[num_cols] = filled_df[num_cols].round(0).astype(int)  # 数值取整，填充数值特征

print("\nMICE多重插补后的后的数据：")
print(df_final)


MICE多重插补后的后的数据：
   用户ID   姓名 性别  年龄         城市     年收入  上次消费金额
0  1001   张三  男  25    beijing   80000     200
1  1002   李四  女  30   shanghai  120000     500
2  1003   王五  男  32  guangzhou   75000     300
3  1004   赵六  女  45   shenzhen   95000     312
4  1005   孙七  男  30   shanghai  120000     500
5  1006   周八  女  28    beijing  110000     400
6  1007   吴九  男  32   hangzhou   98000     250
7  1008   郑十  女  32    nanjing  105000     350
8  1009  钱十一  男  40   shenzhen   88000     280
9  1010  陈十二  女  35    beijing   92000     320


### 5.2 PMM（预测均值匹配，Predictive Mean Matching）

PMM 是非参数插补方法，通过 “预测→匹配→随机选择” 流程为缺失值赋值，核心是保留变量间的非线性与非正态关系，避免模型假设带来的偏差。

In [7]:
import pandas as pd
import numpy as np
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import OneHotEncoder
import warnings
warnings.filterwarnings('ignore')

df_process = df.copy()
# 统一城市大小写（解决 beijing/Beijing 不一致问题）
df_process['城市'] = df_process['城市'].str.lower()

# 划分列类型（精准筛选特征，排除干扰项）
id_cols = ["用户ID", "姓名"]          # 唯一标识（不参与插补）
cat_cols = ["性别", "城市"]           # 分类特征（需编码）
num_cols = ["年龄", "年收入", "上次消费金额"]  # 数值特征（含缺失值）

# 独热编码：无序分类特征转数值（无顺序偏差，保证PMM计算准确）
encoder = OneHotEncoder(sparse_output=False, drop='first')
cat_encoded = encoder.fit_transform(df_process[cat_cols])
cat_df = pd.DataFrame(cat_encoded, columns=encoder.get_feature_names_out(cat_cols))

# 构建PMM插补专用数据集（编码特征 + 数值特征）
pmm_data = pd.concat([cat_df, df_process[num_cols]], axis=1)

# ===================== 4. PMM 预测均值匹配 核心代码 =====================
# 初始化PMM插补器（sklearn标准实现，多重插补+真实值匹配）
pmm_imputer = IterativeImputer(
    random_state=42,        # 固定随机数，结果可复现
    max_iter=50,            # 迭代次数，保证填充收敛
    sample_posterior=True,  # ✅ 核心：开启PM M预测均值匹配模式
    skip_complete=True      # 跳过无缺失值的列，提升效率
)

# 执行PMM多重插补
pmm_filled = pmm_imputer.fit_transform(pmm_data)
filled_df = pd.DataFrame(pmm_filled, columns=pmm_data.columns)

# ===================== 5. 还原原始数据格式（无编码、纯业务字段） =====================
# 逆变换：将编码还原为 性别、城市 原始文字
cat_original = encoder.inverse_transform(filled_df[cat_df.columns])
cat_original_df = pd.DataFrame(cat_original, columns=cat_cols)

# 合并最终数据（完全还原原始格式）
df_final = df.copy()
df_final[cat_cols] = cat_original_df  # 还原分类特征
df_final[num_cols] = filled_df[num_cols].round(0).astype(int)  # 数值取整

print("\nPMM多重插补后的后的数据：")
print(df_final)


PMM多重插补后的后的数据：
   用户ID   姓名 性别  年龄         城市     年收入  上次消费金额
0  1001   张三  男  25    beijing   80000     200
1  1002   李四  女  30   shanghai  120000     500
2  1003   王五  男  14  guangzhou   75000     300
3  1004   赵六  女  45   shenzhen   95000     326
4  1005   孙七  男  30   shanghai  120000     500
5  1006   周八  女  28    beijing  110000     400
6  1007   吴九  男  23   hangzhou   98000     250
7  1008   郑十  女  32    nanjing  105000     350
8  1009  钱十一  男  40   shenzhen   88000     280
9  1010  陈十二  女  35    beijing   92000     320


### 5.3 Bayesian MI（贝叶斯多重插补）

贝叶斯 MI 将缺失值视为待估计的参数，通过 **马尔可夫链蒙特卡洛（MCMC）** 方法从后验分布中采样，获得缺失值的多个合理估计。

In [8]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer        # 多重插补核心
from sklearn.linear_model import BayesianRidge     # 贝叶斯回归模型
from sklearn.preprocessing import OneHotEncoder    # 分类特征编码
import warnings
warnings.filterwarnings('ignore')

df_process = df.copy()
# 统一城市大小写（消除 beijing/Beijing 不一致问题）
df_process['城市'] = df_process['城市'].str.lower()

# 划分列类型：排除无关列 + 分类列 + 数值列
id_cols = ["用户ID", "姓名"]
cat_cols = ["性别", "城市"]       # 无序分类特征
num_cols = ["年龄", "年收入", "上次消费金额"]  # 数值特征（含缺失值）

# 独热编码：适配贝叶斯模型，无顺序偏差
encoder = OneHotEncoder(sparse_output=False, drop='first')
cat_encoded = encoder.fit_transform(df_process[cat_cols])
cat_df = pd.DataFrame(cat_encoded, columns=encoder.get_feature_names_out(cat_cols))

# 构建贝叶斯插补专用数据集
mi_data = pd.concat([cat_df, df_process[num_cols]], axis=1)

# ===================== 3. 贝叶斯多重插补核心代码 =====================
# 初始化插补器：使用贝叶斯线性回归作为估计器
bayesian_imputer = IterativeImputer(
    estimator=BayesianRidge(),  # 贝叶斯回归（核心）
    random_state=42,            # 结果可复现
    max_iter=100,               # 迭代次数
    tol=1e-3                    # 收敛阈值
)

# 执行插补
filled_data = bayesian_imputer.fit_transform(mi_data)
filled_df = pd.DataFrame(filled_data, columns=mi_data.columns)

# ===================== 4. 还原原始数据格式 =====================
# 逆变换：将编码还原为原始文字（性别/城市）
cat_original = encoder.inverse_transform(filled_df[cat_df.columns])
cat_original_df = pd.DataFrame(cat_original, columns=cat_cols)

# 合并最终完整数据
df_final = df.copy()
df_final[cat_cols] = cat_original_df
df_final[num_cols] = filled_df[num_cols].round(0).astype(int)  # 数值取整

print("\nBayesian多重插补后的后的数据：")
print(df_final)


Bayesian多重插补后的后的数据：
   用户ID   姓名 性别  年龄         城市     年收入  上次消费金额
0  1001   张三  男  25    beijing   80000     200
1  1002   李四  女  30   shanghai  120000     500
2  1003   王五  男  36  guangzhou   75000     300
3  1004   赵六  女  45   shenzhen   95000     325
4  1005   孙七  男  30   shanghai  120000     500
5  1006   周八  女  28    beijing  110000     400
6  1007   吴九  男  33   hangzhou   98000     250
7  1008   郑十  女  32    nanjing  105000     350
8  1009  钱十一  男  40   shenzhen   88000     280
9  1010  陈十二  女  35    beijing   92000     320


### 5.4 汇总分析（Pooling）

多重插补的核心优势在于能通过 **Rubin's rules** 汇总多个插补结果，得到综合估计与不确定性度量。这一方法适用于所有多重插补技术，包括MICE、PMM和Bayesian MI。

**Rubin's 规则**详细步骤：

1. **点估计**：各次插补结果的均值
   - 公式：\( \bar{Q} = \frac{1}{m} \sum_{i=1}^{m} \hat{Q}_i \)
   - 其中m为插补次数，\( \hat{Q}_i \)为第i次插补的估计值

2. **组内方差**：单次插补内部的变异（反映数据固有波动）
   - 公式：\( \bar{U} = \frac{1}{m} \sum_{i=1}^{m} U_i \)
   - 其中\( U_i \)为第i次插补结果的方差

3. **组间方差**：不同插补间的变异（反映缺失值不确定性）
   - 公式：\( B = \frac{1}{m-1} \sum_{i=1}^{m} (\hat{Q}_i - \bar{Q})^2 \)

4. **总方差**：综合组内和组间变异
   - 公式：\( T = \bar{U} + (1 + \frac{1}{m})B \)
   - 标准误：\( SE = \sqrt{T} \)

5. **自由度**：使用Rubin's公式计算，反映估计的可靠性

**在三种方法中的应用**：
- **MICE**：通过链式方程生成多个插补数据集，每个数据集都经过多轮迭代收敛
- **PMM**：通过预测均值匹配生成符合原始数据分布的插补值，汇总时能保留变量间的非线性关系
- **Bayesian MI**：通过MCMC采样生成后验分布的多个样本，汇总时能反映参数的不确定性

**实现步骤**：
1. 对每种多重插补方法生成 5-10 个完整数据集
2. 对每个数据集计算统计量（均值、方差等）
3. 按Rubin规则汇总，得到最终估计与标准误
4. 计算95%置信区间，评估估计的可靠性

**适用场景**：需要统计推断（如假设检验、置信区间）的场景，能正确反映缺失值带来的不确定性。


In [9]:
# 生成三种方法的多重插补数据集
def generate_multiple_imputations(df, n_imputations=5):
    """
    生成三种方法的多重插补数据集
    参数：
        df: 原始数据集
        n_imputations: 插补次数
    返回：
        dict: 包含三种方法的插补数据集列表
    """
    # 数据预处理
    df_process = df.copy()
    df_process['城市'] = df_process['城市'].str.lower()
    
    # 分离特征
    cat_cols = ['性别', '城市']
    num_cols = ['年龄', '年收入', '上次消费金额']
    
    # 独热编码分类特征
    encoder = OneHotEncoder(sparse_output=False, drop='first')
    cat_encoded = encoder.fit_transform(df_process[cat_cols])
    cat_df = pd.DataFrame(cat_encoded, columns=encoder.get_feature_names_out(cat_cols))
    
    # 构建建模数据
    model_data = pd.concat([cat_df, df_process[num_cols]], axis=1)
    
    # 存储三种方法的插补结果
    imputed_datasets = {
        'MICE': [],
        'PMM': [],
        'Bayesian': []
    }
    
    # MICE多重插补
    for i in range(n_imputations):
        mice_imputer = IterativeImputer(
            estimator=RandomForestRegressor(random_state=42 + i),
            random_state=42 + i,
            max_iter=10
        )
        mice_filled = mice_imputer.fit_transform(model_data)
        mice_df = pd.DataFrame(mice_filled, columns=model_data.columns)
        # 还原分类特征
        cat_original = encoder.inverse_transform(mice_df[cat_df.columns])
        cat_original_df = pd.DataFrame(cat_original, columns=cat_cols)
        # 构建完整数据集
        result_df = df.copy()
        result_df[cat_cols] = cat_original_df
        result_df[num_cols] = mice_df[num_cols].round(0).astype(int)
        imputed_datasets['MICE'].append(result_df)
    
    # PMM多重插补
    for i in range(n_imputations):
        pmm_imputer = IterativeImputer(
            random_state=42 + i,
            max_iter=50,
            sample_posterior=True,
            skip_complete=True
        )
        pmm_filled = pmm_imputer.fit_transform(model_data)
        pmm_df = pd.DataFrame(pmm_filled, columns=model_data.columns)
        # 还原分类特征
        cat_original = encoder.inverse_transform(pmm_df[cat_df.columns])
        cat_original_df = pd.DataFrame(cat_original, columns=cat_cols)
        # 构建完整数据集
        result_df = df.copy()
        result_df[cat_cols] = cat_original_df
        result_df[num_cols] = pmm_df[num_cols].round(0).astype(int)
        imputed_datasets['PMM'].append(result_df)
    
    # Bayesian MI多重插补
    for i in range(n_imputations):
        bayesian_imputer = IterativeImputer(
            estimator=BayesianRidge(),
            random_state=42 + i,
            max_iter=100,
            tol=1e-3
        )
        bayesian_filled = bayesian_imputer.fit_transform(model_data)
        bayesian_df = pd.DataFrame(bayesian_filled, columns=model_data.columns)
        # 还原分类特征
        cat_original = encoder.inverse_transform(bayesian_df[cat_df.columns])
        cat_original_df = pd.DataFrame(cat_original, columns=cat_cols)
        # 构建完整数据集
        result_df = df.copy()
        result_df[cat_cols] = cat_original_df
        result_df[num_cols] = bayesian_df[num_cols].round(0).astype(int)
        imputed_datasets['Bayesian'].append(result_df)
    
    return imputed_datasets

def pool_multiple_imputations(imputed_datasets, column_name):
    """
    汇总多重插补结果
    参数：
        imputed_datasets: 多个DataFrame组成的列表
        column_name: 要汇总的列名
    返回：
        dict: 包含点估计和标准误的字典
    """
    m = len(imputed_datasets)
    
    imp_values = [df[column_name].values for df in imputed_datasets]
    
    Q_bar = np.mean([v.mean() for v in imp_values])
    
    within_var = np.mean([v.var() for v in imp_values])
    between_var = np.var([v.mean() for v in imp_values])
    
    total_var = within_var + (1 + 1/m) * between_var
    total_std = np.sqrt(total_var)
    
    return {
        'point_estimate': Q_bar,
        'within_variance': within_var,
        'between_variance': between_var,
        'total_variance': total_var,
        'standard_error': total_std,
        'confidence_interval_95': (Q_bar - 1.96 * total_std, Q_bar + 1.96 * total_std)
    }

# 生成多重插补数据集
imputed_datasets = generate_multiple_imputations(df, n_imputations=5)

print("="*60)
print("汇总分析结果（Rubin's rules）")
print("="*60)

# 对每种方法进行汇总分析
for method in ['MICE', 'PMM', 'Bayesian']:
    print(f"{method}方法汇总结果：")
    for col in ['年龄', '上次消费金额']:
        result = pool_multiple_imputations(imputed_datasets[method], col)
        print(f"  【{col}】")
        print(f"    点估计（均值）: {result['point_estimate']:.2f}")
        print(f"    组内方差: {result['within_variance']:.4f}")
        print(f"    组间方差: {result['between_variance']:.4f}")
        print(f"    总方差: {result['total_variance']:.4f}")
        print(f"    标准误: {result['standard_error']:.2f}")
        print(f"    95%置信区间: ({result['confidence_interval_95'][0]:.2f}, {result['confidence_interval_95'][1]:.2f})")


汇总分析结果（Rubin's rules）
MICE方法汇总结果：
  【年龄】
    点估计（均值）: 32.90
    组内方差: 30.7620
    组间方差: 0.0080
    总方差: 30.7716
    标准误: 5.55
    95%置信区间: (22.03, 43.77)
  【上次消费金额】
    点估计（均值）: 340.76
    组内方差: 8925.6440
    组间方差: 0.1384
    总方差: 8925.8101
    标准误: 94.48
    95%置信区间: (155.59, 525.93)
PMM方法汇总结果：
  【年龄】
    点估计（均值）: 31.12
    组内方差: 63.8440
    组间方差: 1.4616
    总方差: 65.5979
    标准误: 8.10
    95%置信区间: (15.25, 46.99)
  【上次消费金额】
    点估计（均值）: 342.96
    组内方差: 9080.5560
    组间方差: 28.7224
    总方差: 9115.0229
    标准误: 95.47
    95%置信区间: (155.83, 530.09)
Bayesian方法汇总结果：
  【年龄】
    点估计（均值）: 33.40
    组内方差: 31.2400
    组间方差: 0.0000
    总方差: 31.2400
    标准误: 5.59
    95%置信区间: (22.45, 44.35)
  【上次消费金额】
    点估计（均值）: 342.50
    组内方差: 8836.2500
    组间方差: 0.0000
    总方差: 8836.2500
    标准误: 94.00
    95%置信区间: (158.26, 526.74)


### 5.5 不同方法对比

以下是MICE、PMM和Bayesian MI三种多重插补方法的对比分析：

**方法特点对比**：
| 方法 | 核心原理 | 优势 | 劣势 |
|------|----------|------|------|
| MICE | 链式方程迭代预测 | 考虑变量间关系，适用于多种数据类型 | 计算开销较大，需要多轮迭代 |
| PMM | 预测均值匹配 | 保留数据分布，避免模型假设偏差 | 实现复杂度较高 |
| Bayesian MI | MCMC后验采样 | 反映参数不确定性，理论基础扎实 | 计算时间长，对小样本敏感 |

**插补结果对比**：
- **MICE**：插补值平滑，能捕捉变量间的线性关系
- **PMM**：插补值更接近原始数据分布，适合非正态数据
- **Bayesian MI**：插补值具有不确定性，能反映参数的概率分布

**适用场景推荐**：
- **MICE**：一般情况下的首选方法，适用于大多数数据集
- **PMM**：当需要保留数据原始分布时使用，如分类变量或偏态数据
- **Bayesian MI**：当需要严格的统计推断和不确定性量化时使用


In [10]:
print("="*70)
print("各插补方法结果对比（年龄列）")
print("="*70)

# 提取每种方法的第一个插补结果进行对比
df_compare = pd.DataFrame({
    '用户ID': [1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008, 1009, 1010],
    '原始年龄': [25, 30, np.nan, 45, 30, 28, np.nan, 32, 40, 35],
    'MICE插补': imputed_datasets['MICE'][0]['年龄'].values,
    'PMM插补': imputed_datasets['PMM'][0]['年龄'].values,
    'Bayesian插补': imputed_datasets['Bayesian'][0]['年龄'].values
})

print(df_compare.to_string(index=False))


print("各方法统计量对比")
print("="*70)

# 计算每种方法的统计量
stats_comparison = []

# 原始数据统计量
original_stats = {
    '方法': '原始数据(排除NaN)',
    '均值': df['年龄'].mean(),
    '标准差': df['年龄'].std(),
    '中位数': df['年龄'].median()
}
stats_comparison.append(original_stats)

# 每种方法的统计量
for method in ['MICE', 'PMM', 'Bayesian']:
    # 汇总所有插补结果
    all_ages = []
    for imp_df in imputed_datasets[method]:
        all_ages.extend(imp_df['年龄'].values)
    
    method_stats = {
        '方法': method,
        '均值': np.mean(all_ages),
        '标准差': np.std(all_ages),
        '中位数': np.median(all_ages)
    }
    stats_comparison.append(method_stats)

stats_df = pd.DataFrame(stats_comparison)
print(stats_df.to_string(index=False))


各插补方法结果对比（年龄列）
 用户ID  原始年龄  MICE插补  PMM插补  Bayesian插补
 1001  25.0      25     25          25
 1002  30.0      30     30          30
 1003   NaN      32     14          36
 1004  45.0      45     45          45
 1005  30.0      30     30          30
 1006  28.0      28     28          28
 1007   NaN      32     23          33
 1008  32.0      32     32          32
 1009  40.0      40     40          40
 1010  35.0      35     35          35
各方法统计量对比
         方法     均值      标准差  中位数
原始数据(排除NaN) 33.125 6.599513 31.0
       MICE 32.900 5.547071 32.0
        PMM 31.120 8.081188 30.0
   Bayesian 33.400 5.589275 32.5
